# 02 — Conformer Generation

For every molecule in `data/splits.json` (train+valid+test):
1. Embed multiple 3D conformers with ETKDGv3.
2. Optimize each with MMFF94 (fallback UFF if MMFF params are missing for some atom type).
3. Convert absolute energies to relative energies and Boltzmann weights at T=298.15K.
4. Cache atomic numbers + coordinates + energies + weights to `data/processed/conformers/<id>.npz`.

In [2]:
import sys
sys.path.append("..")

import time
from pathlib import Path

import numpy as np
import yaml
from tqdm.auto import tqdm

from src.utils import load_json, save_json, set_seed
from src.conformers import generate_conformers

with open("../config/conformers.yaml") as f:
    cfg = yaml.safe_load(f)

set_seed(cfg["conformers"]["random_seed"])
cfg


c:\Users\ajaya\anaconda3\envs\chemprop\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'conformers': {'n_confs': 8,
  'prune_rms_thresh': 0.5,
  'max_iters': 500,
  'forcefield': 'MMFF94',
  'energy_window_kcal': 10.0,
  'random_seed': 42},
 'boltzmann': {'temperature_K': 298.15},
 'paths': {'splits_file': '../data/splits.json',
  'out_dir': '../data/processed/conformers',
  'manifest_file': '../data/processed/conformer_manifest.json'}}

In [3]:
splits = load_json(cfg["paths"]["splits_file"])

out_dir = Path(cfg["paths"]["out_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

# Build one flat worklist across all three splits, deduplicated by id
# (a molecule shouldn't appear in more than one split under a scaffold split, but dedupe defensively anyway)
worklist = {}
for split_name in ["train", "valid", "test"]:
    for mol_id, smi in zip(splits[split_name]["id"], splits[split_name]["smiles"]):
        worklist[mol_id] = smi

print(f"Total unique molecules to process: {len(worklist)}")


Total unique molecules to process: 616


## Quick timing estimate on a small sample

In [4]:
sample_ids = list(worklist.keys())[:10]
t0 = time.time()
for mol_id in sample_ids:
    generate_conformers(worklist[mol_id], **cfg["conformers"])
elapsed = time.time() - t0

per_mol = elapsed / len(sample_ids)
est_total_min = per_mol * len(worklist) / 60
print(f"~{per_mol:.2f} sec/molecule -> estimated total: {est_total_min:.1f} minutes for {len(worklist)} molecules")


~0.23 sec/molecule -> estimated total: 2.4 minutes for 616 molecules


In [7]:
manifest = {"success": [], "failed": [], "forcefield_used": {}, "n_confs_kept": {}}

t_start = time.time()
for mol_id, smi in tqdm(worklist.items(), total=len(worklist)):
    out_path = out_dir / f"{mol_id}.npz"
    if out_path.exists():
        manifest["success"].append(mol_id)
        cached = np.load(out_path)
        manifest["n_confs_kept"][mol_id] = int(cached["coords"].shape[0])
        continue

    result = generate_conformers(smi, **cfg["conformers"])
    if result is None:
        manifest["failed"].append({"id": mol_id, "smiles": smi})
        continue

    np.savez_compressed(
        out_path,
        atomic_nums=result["atomic_nums"],
        coords=result["coords"],
        energies_kcal=result["energies_kcal"],
        boltzmann_weights=result["boltzmann_weights"],
    )
    manifest["success"].append(mol_id)
    manifest["forcefield_used"][mol_id] = result["forcefield_used"]
    manifest["n_confs_kept"][mol_id] = result["n_confs_kept"]

elapsed_min = (time.time() - t_start) / 60
print(f"Done in {elapsed_min:.1f} min. Success: {len(manifest['success'])}  Failed: {len(manifest['failed'])}")


100%|██████████| 616/616 [05:10<00:00,  1.98it/s]

Done in 5.2 min. Success: 612  Failed: 4


## Inspect failures and forcefield fallback rate

In [8]:
print(f"Failed: {len(manifest['failed'])} / {len(worklist)}")
for f in manifest["failed"][:10]:
    print(f['id'], f['smiles'])

ff_counts = {}
for ff in manifest["forcefield_used"].values():
    ff_counts[ff] = ff_counts.get(ff, 0) + 1
print("Forcefield usage:", ff_counts)

n_confs_kept = list(manifest["n_confs_kept"].values())
print(f"Conformers kept per molecule: mean={np.mean(n_confs_kept):.2f}, min={np.min(n_confs_kept)}, max={np.max(n_confs_kept)}")


Failed: 4 / 616
BMCL-16-2006-4263-1 CN1CCN(CCCN(C(=O)Nc2ccc(F)c(C(F)(F)F)c2)[C@@H]2C[C@@H](c3ccc(C#N)cc3)[C@@H]3C[C@H]32)CC1
BMCL-16-2006-4265_26 CN1CCN(CCCN(C(=O)Nc2ccc(F)c(C(F)(F)F)c2)[C@@H]2C[C@@H](c3ccc4c(c3)[C@H](N)NO4)[C@@H]3C[C@H]32)CC1
AJMALINE CC[C@H]1[C@H]2C[C@H]3[C@@H]4N(C)c5ccccc5[C@]45C[C@@H]([C@H]2[C@H]5O)N3[C@@H]1O
METHYLECGONIDINE COC(=O)C1=CC[C@@H]2CC[C@@H]1[NH+]2C
Forcefield usage: {}
Conformers kept per molecule: mean=5.34, min=1, max=8


In [9]:
save_json(manifest, cfg["paths"]["manifest_file"])
print("Saved manifest:", cfg["paths"]["manifest_file"])


Saved manifest: ../data/processed/conformer_manifest.json


## Sanity-check one molecule's Boltzmann weights

Lower relative energy should always map to higher weight — a quick, direct check that the physics is wired correctly before it feeds the 3D encoder.

In [14]:
check_id = manifest["success"][0]
d = np.load(out_dir / f"{check_id}.npz")

print("Relative energies (kcal/mol):", d["energies_kcal"])
print("Boltzmann weights:          ", d["boltzmann_weights"])
print("Weights sum to:", d["boltzmann_weights"].sum())
assert np.argmin(d["energies_kcal"]) == np.argmax(d["boltzmann_weights"]), "lowest-energy conformer should get the highest weight"
print("OK — lowest-energy conformer has the highest Boltzmann weight, as expected.")


Relative energies (kcal/mol): [0.5588129 2.6750875 2.2202551 3.6051483 2.6853945 0.        3.790185
 2.7946637]
Boltzmann weights:           [0.2689996  0.00755982 0.01628943 0.00157316 0.00742945 0.69081914
 0.00115117 0.00617819]
Weights sum to: 0.99999994
OK — lowest-energy conformer has the highest Boltzmann weight, as expected.


In [15]:
# Drop failed molecules from all splits for consistency across every ablation
dropped_ids = {f["id"] for f in manifest["failed"]}

splits_filtered = {"dataset": splits["dataset"], "dev_seed": splits["dev_seed"]}
for split_name in ["train", "valid", "test"]:
    keep_mask = [mid not in dropped_ids for mid in splits[split_name]["id"]]
    splits_filtered[split_name] = {
        key: [v for v, keep in zip(vals, keep_mask) if keep]
        for key, vals in splits[split_name].items()
    }
    print(f"{split_name}: {len(splits[split_name]['id'])} -> {len(splits_filtered[split_name]['id'])}")

save_json(manifest["failed"], "../data/processed/dropped_ids.json")  # keep the reason on record for the report
save_json(splits_filtered, cfg["paths"]["splits_file"])
print(f"Dropped {len(dropped_ids)} molecules. splits.json updated in place.")

train: 457 -> 453
valid: 66 -> 66
test: 132 -> 132
Dropped 4 molecules. splits.json updated in place.
